In [8]:
import os
import json
import pandas as pd

from openai import OpenAI

from retrieval_module import (
    build_retrieval_system,
    retrieve
)

In [ ]:
print("Loading Retrieval System...")

df, emb_model, faiss_index, bm25 = build_retrieval_system()

print("Retrieval System Ready")

Loading Retrieval System...
[INFO] Chargement du dataset : c:\Users\Hp\Desktop\nlp2\master_dataset_preprocessed.xlsx
[INFO] Dataset chargé : 3038 lignes, colonnes : ['text', 'sender', 'label', 'channel', 'language', 'attack_type', 'risk_level', 'preprocessed_text']
[INFO] Après nettoyage NaN : 3038 lignes
[INFO] Chargement des embeddings depuis c:\Users\Hp\Desktop\nlp2\embeddings.npy
[INFO] Chargement de l'index FAISS : c:\Users\Hp\Desktop\nlp2\faiss_index.bin
[INFO] Chargement de l'index BM25 : c:\Users\Hp\Desktop\nlp2\bm25_index.pkl


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



[INFO] Système de retrieval prêt ✓
Retrieval System Ready


In [10]:
query = "Your account has been suspended. Click here immediately to verify your account."

retrieved = retrieve(
    query,
    emb_model,
    faiss_index,
    bm25,
    df,
    top_k=5
)

print("Nombre de résultats :", len(retrieved))

retrieved[[
    "text",
    "label",
    "language",
    "attack_type"
]]

Nombre de résultats : 5


,text,label,language,attack_type
0,Unusual activity has been detected on your Ban...,phishing,en,credential_theft
1,Your email account has been inactive and is sc...,phishing,en,credential_theft
2,تم تعليق حسابك البنكي مؤقتاً. حدث بياناتك فورا...,phishing,ar,credential_theft
3,A new payment has been received in your Payone...,phishing,en,credential_theft
4,ALERT: Your Uber account will be suspended. Ve...,phishing,en,urgency


In [11]:
def build_context(retrieved_df):

    context_parts = []

    for i, (_, row) in enumerate(
        retrieved_df.iterrows(),
        start=1
    ):

        context_parts.append(
            f"""
Example {i}

Text:
{row['text']}

Label:
{row['label']}

Language:
{row['language']}

Attack Type:
{row['attack_type']}

Risk Level:
{row['risk_level']}
"""
        )

    return "\n".join(context_parts)
context = build_context(retrieved)

print(context)


Example 1

Text:
Unusual activity has been detected on your Bank of America account your account has been temporarily suspended

Label:
phishing

Language:
en

Attack Type:
credential_theft

Risk Level:
critical


Example 2

Text:
Your email account has been inactive and is scheduled for deletion please verify your account within 24 hours to prevent deactivation

Label:
phishing

Language:
en

Attack Type:
credential_theft

Risk Level:
critical


Example 3

Text:
تم تعليق حسابك البنكي مؤقتاً. حدث بياناتك فوراً عبر الرابط.

Label:
phishing

Language:
ar

Attack Type:
credential_theft

Risk Level:
critical


Example 4

Text:
A new payment has been received in your Payoneer account please verify your account immediately to release the funds

Label:
phishing

Language:
en

Attack Type:
credential_theft

Risk Level:
critical


Example 5

Text:
ALERT: Your Uber account will be suspended. Verify: uber-check.biz/verify

Label:
phishing

Language:
en

Attack Type:
urgency

Risk Level:
critical

In [ ]:
import os

client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

LLM_MODEL = "llama-3.3-70b-versatile"

print("LLM Ready")
print("Model :", LLM_MODEL)

LLM Ready
Model : llama-3.3-70b-versatile


In [ ]:
import re
import json

def extract_json(raw):

    if not raw:
        return None

    match = re.search(
        r"\{.*\}",
        raw,
        re.DOTALL
    )

    if match:
        return match.group(0)

    return None

In [14]:
def rag_analyze(query: str, top_k: int = 5):

    # 1. Retrieval
    retrieved = retrieve(
        query,
        emb_model,
        faiss_index,
        bm25,
        df,
        top_k=top_k
    )

    # 2. Context
    context = build_context(retrieved)

    # 3. Prompt
    prompt = f"""
You are a cybersecurity expert specialized in phishing detection.

Use the retrieved examples as context.

Retrieved Examples:

{context}

Analyze the following message.

Message:
{query}

Return ONLY valid JSON.
Important: if label is legitimate, risk_score must be between 0 and 30.

{{
  "label": "phishing|legitimate",
  "confidence": <float 0.0-1.0>,
  "risk_score": <int 0-100>,
  "attack_type": "credential_theft|financial_scam|malicious_link|impersonation|urgency|none",
  "explanation": "short explanation",
  "suspicious_elements": ["item1", "item2"],
  "recommendation": "recommended action"
}}
"""

    try:

        response = client.chat.completions.create(
            model=LLM_MODEL,
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            temperature=0,
            max_tokens=400
        )

        raw = response.choices[0].message.content

        json_text = extract_json(raw)

        if json_text is None:
            raise ValueError(
                "No JSON found in model response"
            )

        parsed = json.loads(json_text)

        # Confidence normalization
        confidence = parsed.get(
            "confidence",
            0.0
        )

        if isinstance(confidence, str):

            mapping = {
                "low": 0.3,
                "medium": 0.6,
                "high": 0.9
            }

            confidence = mapping.get(
                confidence.lower(),
                0.0
            )

        try:
            confidence = float(confidence)
        except:
            confidence = 0.0

        confidence = max(
            0.0,
            min(1.0, confidence)
        )

        parsed["confidence"] = confidence

       

        parsed["retrieved_examples"] = len(
            retrieved
        )

        return parsed

    except Exception as e:

        return {
            "label": "error",
            "error": str(e)
        }

In [15]:
test_query = """
Your account has been suspended.
Click here immediately to verify your account.
"""

result = rag_analyze(test_query)

print(result)

{'label': 'phishing', 'confidence': 0.95, 'risk_score': 90, 'attack_type': 'credential_theft', 'explanation': 'The message creates a sense of urgency and prompts the user to click on a link to verify their account, which is a common phishing tactic.', 'suspicious_elements': ['unsolicited account suspension', 'request to click on a link'], 'recommendation': 'Do not click on the link and instead contact the service provider directly to verify the account status.', 'retrieved_examples': 5}


In [16]:
result = rag_analyze(
    "Bonjour, votre colis est arrivé. Veuillez confirmer votre identité via ce lien."
)

print(result)

{'label': 'phishing', 'confidence': 0.9, 'risk_score': 90, 'attack_type': 'credential_theft', 'explanation': 'The message creates a sense of urgency and asks the recipient to confirm their identity via a link, which is a common phishing tactic.', 'suspicious_elements': ['urgent tone', 'request for identity confirmation via link'], 'recommendation': 'Do not click on the link and verify the authenticity of the message with the shipping company directly.', 'retrieved_examples': 5}


In [17]:
result = rag_analyze(
    "تم تعليق حسابك البنكي مؤقتًا. اضغط على الرابط لتحديث بياناتك فورًا."
)

print(result)

{'label': 'phishing', 'confidence': 0.9, 'risk_score': 80, 'attack_type': 'urgency', 'explanation': 'The message creates a sense of urgency by stating the account is suspended and requires immediate action.', 'suspicious_elements': ['تم تعليق حسابك البنكي مؤقتًا', 'اضغط على الرابط لتحديث بياناتك فورًا'], 'recommendation': 'Do not click on the link and contact the bank directly to verify the authenticity of the message.', 'retrieved_examples': 5}


In [18]:
result = rag_analyze(
    "Hello Imad, the meeting is scheduled for tomorrow at 10 AM in room B12."
)

print(result)

{'label': 'legitimate', 'confidence': 0.9, 'risk_score': 10, 'attack_type': 'none', 'explanation': 'The message appears to be a genuine meeting reminder, similar to the retrieved examples, with no suspicious requests or links.', 'suspicious_elements': [], 'recommendation': 'No action required, proceed with the meeting as scheduled.', 'retrieved_examples': 5}


In [19]:
result = rag_analyze(
    "Bonjour Imad, la réunion est prévue demain à 10h dans la salle B12."
)

print(result)

{'label': 'legitimate', 'confidence': 0.9, 'risk_score': 10, 'attack_type': 'none', 'explanation': 'The message appears to be a genuine meeting reminder, with a specific time and location.', 'suspicious_elements': [], 'recommendation': 'No action required, the message is likely legitimate.', 'retrieved_examples': 5}


In [20]:
result = rag_analyze(
    "مرحباً عماد، الاجتماع سيكون غداً على الساعة العاشرة صباحاً في القاعة B12."
)

print(result)

{'label': 'legitimate', 'confidence': 0.9, 'risk_score': 10, 'attack_type': 'none', 'explanation': 'The message appears to be a genuine meeting invitation with no suspicious elements or requests.', 'suspicious_elements': [], 'recommendation': 'No action required, the message is likely a legitimate meeting invitation.', 'retrieved_examples': 5}


In [21]:
import re
from urllib.parse import urlparse

In [22]:
SUSPICIOUS_WORDS = [
    "verify", "secure", "login", "account", "update",
    "click", "confirm", "banking", "paypal", "amazon",
    "apple", "microsoft", "support", "suspend", "urgent",
    "free", "winner", "prize", "password", "credential"
]

SUSPICIOUS_TLDS = [".xyz", ".ru", ".tk", ".ml", ".ga", ".cf", ".gq", ".top"]

def extract_url_features(url: str) -> dict:
    parsed = urlparse(url)
    
    domain    = parsed.netloc
    path      = parsed.path
    scheme    = parsed.scheme
    
    # Sous-domaines
    parts     = domain.split(".")
    subdomain = ".".join(parts[:-2]) if len(parts) > 2 else ""
    tld       = "." + parts[-1] if parts else ""
    
    # Features
    features = {
        "url"              : url,
        "domain"           : domain,
        "subdomain"        : subdomain,
        "path"             : path,
        "scheme"           : scheme,
        "url_length"       : len(url),
        "has_ip"           : bool(re.match(r"^\d{1,3}(\.\d{1,3}){3}$", domain)),
        "has_at"           : "@" in url,
        "dash_count"       : domain.count("-"),
        "subdomain_count"  : len(parts) - 2 if len(parts) > 2 else 0,
        "is_https"         : scheme == "https",
        "suspicious_tld"   : tld in SUSPICIOUS_TLDS,
        "suspicious_words" : [w for w in SUSPICIOUS_WORDS if w in url.lower()],
    }
    
    return features

In [23]:
def features_to_text(features: dict) -> str:
    lines = [
        f"URL analysée : {features['url']}",
        f"Domaine : {features['domain']}",
        f"Sous-domaine : {features['subdomain'] or 'aucun'}",
        f"Chemin : {features['path'] or '/'}",
        f"Protocole : {features['scheme'].upper()}",
        f"Longueur totale : {features['url_length']} caractères",
        f"Contient une IP : {'oui' if features['has_ip'] else 'non'}",
        f"Contient @ : {'oui' if features['has_at'] else 'non'}",
        f"Nombre de tirets dans le domaine : {features['dash_count']}",
        f"Nombre de sous-domaines : {features['subdomain_count']}",
        f"HTTPS : {'oui' if features['is_https'] else 'non'}",
        f"Extension suspecte : {'oui' if features['suspicious_tld'] else 'non'}",
        f"Mots suspects détectés : {', '.join(features['suspicious_words']) if features['suspicious_words'] else 'aucun'}",
    ]
    return "\n".join(lines)

In [24]:
def analyze_url(url: str) -> dict:
    # Etape 1 : extraire les features
    features = extract_url_features(url)
    
    # Etape 2 : convertir en texte
    text = features_to_text(features)
    
    # Etape 3 : passer au RAG
    result = rag_analyze(text)
    
    # Etape 4 : ajouter les features au résultat
    result["url_features"] = features
    
    return result

In [ ]:
test_urls = [
    "http://paypal-secure-login.xyz/verify/account",
    "https://www.google.com",
    "http://192.168.1.1/login?user=admin",
    "https://amazon-update-account.tk/confirm",
    "https://visacru.gu.cc/home.html",
    "https://tiny1.org/1v"

]

for url in test_urls:
    print("\n" + "="*60)
    print(f"URL : {url}")
    result = analyze_url(url)
    print(f"Label      : {result.get('label')}")
    print(f"Confidence : {result.get('confidence')}")
    print(f"Risk Score : {result.get('risk_score')}")
    print(f"Attack     : {result.get('attack_type')}")
    print(f"Explanation: {result.get('explanation')}")


URL : http://paypal-secure-login.xyz/verify/account
Label      : phishing
Confidence : 0.95
Risk Score : 90
Attack     : credential_theft
Explanation: The URL contains suspicious keywords like 'verify', 'secure', 'login', and 'account', and the domain 'paypal-secure-login.xyz' is likely a typosquatting attempt to impersonate PayPal.

URL : https://www.google.com
Label      : legitimate
Confidence : 0.99
Risk Score : 5
Attack     : none
Explanation: The URL is a well-known and trusted domain (google.com) with no suspicious elements detected.

URL : http://192.168.1.1/login?user=admin
Label      : phishing
Confidence : 0.9
Risk Score : 80
Attack     : credential_theft
Explanation: The URL contains a login path and an IP address, which could be used to steal credentials

URL : https://amazon-update-account.tk/confirm
Label      : phishing
Confidence : 0.95
Risk Score : 90
Attack     : impersonation
Explanation: The URL is impersonating Amazon, with suspicious words 'account', 'update', a